In [17]:
import pandas as pd



df_original = pd.read_csv("healthcare_dataset.csv")

print(f"✅ Data Loaded: {df_original.shape[0]} rows, {df_original.shape[1]} columns")
df_original.head()

✅ Data Loaded: 55500 rows, 15 columns


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [4]:
# ── Missing Values ─────────────────────────────────────
print( df_original.isnull().sum())


Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64


In [5]:
# ── Duplicates ─────────────────────────────────────────
duplicates = df_original.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")

# Remove duplicates if any
df_clean = df_original.drop_duplicates()
print(f"✅ Shape after removing duplicates: {df_clean.shape}")

Total duplicate rows: 534
✅ Shape after removing duplicates: (54966, 15)


In [6]:
# ── Fix Patient Names (Bobby JacksOn → Bobby Jackson) ──
df_clean = df_clean.copy()
df_clean["Name"] = df_clean["Name"].str.title()
print("✅ Names cleaned!")
print(df_clean["Name"].head())

✅ Names cleaned!
0    Bobby Jackson
1     Leslie Terry
2      Danny Smith
3     Andrew Watts
4    Adrienne Bell
Name: Name, dtype: str


In [8]:
# ── Check unique values in each category ───────────────
categorical_cols = [
    "Gender", "Medical Condition",
    "Admission Type", "Insurance Provider",
    "Blood Type", "Medication", "Test Results"
]

for col in categorical_cols:
    print(f"\n{col} → {df_clean[col].nunique()} unique values:")
    print(df_clean[col].unique())


Gender → 2 unique values:
<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

Medical Condition → 6 unique values:
<ArrowStringArray>
['Cancer', 'Obesity', 'Diabetes', 'Asthma', 'Hypertension', 'Arthritis']
Length: 6, dtype: str

Admission Type → 3 unique values:
<ArrowStringArray>
['Urgent', 'Emergency', 'Elective']
Length: 3, dtype: str

Insurance Provider → 5 unique values:
<ArrowStringArray>
['Blue Cross', 'Medicare', 'Aetna', 'UnitedHealthcare', 'Cigna']
Length: 5, dtype: str

Blood Type → 8 unique values:
<ArrowStringArray>
['B-', 'A+', 'A-', 'O+', 'AB+', 'AB-', 'B+', 'O-']
Length: 8, dtype: str

Medication → 5 unique values:
<ArrowStringArray>
['Paracetamol', 'Ibuprofen', 'Aspirin', 'Penicillin', 'Lipitor']
Length: 5, dtype: str

Test Results → 3 unique values:
<ArrowStringArray>
['Normal', 'Inconclusive', 'Abnormal']
Length: 3, dtype: str


In [9]:
# ── Remove leading/trailing spaces ─────────────────────
df_clean = df_clean.copy()
for col in categorical_cols:
    df_clean[col] = df_clean[col].str.strip()

print("✅ Whitespace removed from all text columns!")

✅ Whitespace removed from all text columns!


In [12]:
# ── Convert dates to proper datetime format ─────────────
df_clean["Date of Admission"] = pd.to_datetime(df_clean["Date of Admission"])
df_clean["Discharge Date"]    = pd.to_datetime(df_clean["Discharge Date"])

print("✅ Dates converted!")
print(df_clean[["Date of Admission", "Discharge Date"]].dtypes)
print(df_clean[["Date of Admission", "Discharge Date"]].head())

✅ Dates converted!
Date of Admission    datetime64[us]
Discharge Date       datetime64[us]
dtype: object
  Date of Admission Discharge Date
0        2024-01-31     2024-02-02
1        2019-08-20     2019-08-26
2        2022-09-22     2022-10-07
3        2020-11-18     2020-12-18
4        2022-09-19     2022-10-09


In [13]:
# ── Check for negative or zero values ──────────────────
print("Age range:", df_clean["Age"].min(), "→", df_clean["Age"].max())
print("Billing range:", df_clean["Billing Amount"].min(), "→", df_clean["Billing Amount"].max())

# Remove invalid ages
df_clean = df_clean[(df_clean["Age"] > 0) & (df_clean["Age"] <= 120)]

# Remove negative billing
df_clean = df_clean[df_clean["Billing Amount"] > 0]

print(f"\n✅ Shape after cleaning invalid values: {df_clean.shape}")

Age range: 13 → 89
Billing range: -2008.4921398591305 → 52764.276736469175

✅ Shape after cleaning invalid values: (54860, 15)


In [14]:
# ── Outlier Detection using IQR ─────────────────────────
Q1 = df_clean["Billing Amount"].quantile(0.25)
Q3 = df_clean["Billing Amount"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean["Billing Amount"] < lower) |
    (df_clean["Billing Amount"] > upper)
]

print(f"Outliers found: {len(outliers)}")
print(f"Lower bound: ${lower:,.2f}")
print(f"Upper bound: ${upper:,.2f}")

Outliers found: 0
Lower bound: $-23,521.23
Upper bound: $74,668.04


In [15]:
# ── Final Summary ───────────────────────────────────────
print("=" * 45)
print("      ✅ DATA CLEANING COMPLETE!")
print("=" * 45)
print(f"Original shape : {df_original.shape}")
print(f"Cleaned shape  : {df_clean.shape}")
print(f"Rows removed   : {df_original.shape[0] - df_clean.shape[0]}")
print(f"Missing values : {df_clean.isnull().sum().sum()}")
print(f"Duplicates     : {df_clean.duplicated().sum()}")
print("=" * 45)
df_clean.head()

      ✅ DATA CLEANING COMPLETE!
Original shape : (55500, 15)
Cleaned shape  : (54860, 15)
Rows removed   : 640
Missing values : 0
Duplicates     : 0


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [16]:
df_clean.to_csv("healthcare_cleaned.csv", index=False)
print("✅ Cleaned file saved as healthcare_cleaned.csv")

✅ Cleaned file saved as healthcare_cleaned.csv
